## Project Restructure: Descriptive Analysis

**Notes**
3 Main areas of analysis:
 * Temporal
 * Institutional
 * Geographic

Did not include all territories and jurisdictions that analysis will be done separately. Focusing solely on the U.S. states.

## Temporal Analysis

**Question**
How has undergraduate enrollment changed across U.S. states from 2012-2022, and are these changes statistically significant?

In [199]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t
import plotly.express as px
import plotly.graph_objects as go
import os

In [200]:
enrollment_df = pd.read_csv(os.path.dirname(os.getcwd()) + '/data/cleaned_enrollmentdata.csv')
enrollment_df.rename(columns={'State or jurisdiction': 'State'}, inplace=True)
enrollment_df.head()

,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442


In [201]:
states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
    "Wisconsin", "Wyoming", "District of Columbia"
]

state_enrollment = enrollment_df[enrollment_df["State"].isin(states)]
state_enrollment['State'].unique()

array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District of Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
       'New Jersey', 'New Mexico', 'New York', 'North Carolina',
       'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

--------

**Sub-Question** What is the national enrollment trend when aggregating across all states?

In [202]:
state_enrollment['Total_Enrollment'] = state_enrollment['Total_Pub_Under'] + state_enrollment['Total_Priv_Under']
state_enrollment.head()

/var/folders/6g/_wpr8bs53rs84lqb6nn_rk900000gn/T/ipykernel_48341/1575148348.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc,Total_Enrollment
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967,265917
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043,261188
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513,259630
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233,257650
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442,258008


In [203]:
nationwide_trends = state_enrollment.groupby('Year').agg({'Total_Enrollment': 'sum', 'Total_Pub_Under': 'sum', 'Total_Priv_Under': 'sum'}).reset_index()
nationwide_trends

,Year,Total_Enrollment,Total_Pub_Under,Total_Priv_Under
0,2012-13,17717229,13458541,4258688
1,2013-14,17459865,13332032,4127833
2,2014-15,17278075,13230125,4047950
3,2015-16,17021992,13130934,3891058
4,2016-17,16854168,13126042,3728126
5,2017-18,16745071,13085693,3659378
6,2018-19,16594731,13033822,3560909
7,2019-20,16549691,12986168,3563523
8,2020-21,15836385,12305625,3530760
9,2021-22,15433062,11929275,3503787


In [204]:
baseline_year = nationwide_trends[nationwide_trends['Year'] == '2012-13']
baseline_enrollment = nationwide_trends['Total_Enrollment'][0]
nationwide_trends['enrollment_index'] = (nationwide_trends['Total_Enrollment'] / baseline_enrollment) * 100
nationwide_trends['yoy_change'] = nationwide_trends['Total_Enrollment'].pct_change() * 100

In [205]:
total_change_pct = ((nationwide_trends['Total_Enrollment'][9] - nationwide_trends['Total_Enrollment'][0])/baseline_enrollment) * 100
print(total_change_pct)

-12.892349023653754


In [206]:
years = np.arange(len(nationwide_trends))
slope,intercept, r_value, p_value, std_err = stats.linregress(years, nationwide_trends['enrollment_index'])

In [207]:
n = len(years)
dof = n - 2
t_crit = t.ppf(0.975, dof)  # 95% CI
ci_slope = t_crit * std_err

In [208]:
print('Baseline: ', baseline_enrollment)
print('2021-22: ', nationwide_trends['Total_Enrollment'][9])
print('Total Change %: ', total_change_pct)
print('Avg Change %: ', nationwide_trends['yoy_change'].mean())
print('Slope: ', slope)
print('Intercept: ', intercept)
print('95% CI: ', ci_slope)
print('R Squared: ', r_value**2)
print('P-Value: ', p_value)

Baseline:  17717229
2021-22:  15433062
Total Change %:  -12.892349023653754
Avg Change %:  -1.5150317578493226
Slope:  -1.2641234955277412
Intercept:  100.22380669942542
95% CI:  0.2766016400003222
R Squared:  0.9328116061637491
P-Value:  5.728672704293977e-06


P-Value is much less than the alpha value of 0.05, this means that the national enrollment trend is statistically significant.

### Visualization

In [209]:
## Enrollment Over time
px.line(nationwide_trends, x='Year', y='Total_Enrollment')

In [210]:
## YoY Change Over time
px.line(nationwide_trends, x='Year', y='yoy_change')

In [211]:
## Statistical Predictions
fig = go.Figure()

fig.add_trace(go.Scatter(
    x = nationwide_trends['Year'],
    y = nationwide_trends['enrollment_index'],
    mode = 'lines',
    name = 'Enrollment Index'
))

predictions = slope * years + intercept
fig.add_trace(go.Scatter(
    x=nationwide_trends['Year'],
    y=predictions,
    mode='lines',
    name=f'Linear Trend (R_squared={r_value**2:.3f})',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title='Enrollment Index Over Time',
    xaxis_title='Year',
    yaxis_title='Enrollment Index',
    showlegend=True
)


fig.show()

Baseline is at 100. We see a clear downward trend of enrollment. The linear regression easily captures 93 % of the variance.

**Question** What is the national enrollment trend when aggregating across all states?

**Answer** There is a negative enrollment trend when aggregfating across all the states.

---------

**Question** Which states have statistically significant declining trends?

In [212]:
state_trends = state_enrollment.groupby(['State', 'Year']).agg({'Total_Enrollment': 'sum', 'Total_Pub_Under': 'sum', 'Total_Priv_Under': 'sum'}).reset_index()
state_trends

,State,Year,Total_Enrollment,Total_Pub_Under,Total_Priv_Under
0,Alabama,2012-13,265917,216535,49382
1,Alabama,2013-14,261188,213669,47519
2,Alabama,2014-15,259630,212458,47172
3,Alabama,2015-16,257650,212968,44682
4,Alabama,2016-17,258008,216115,41893
...,...,...,...,...,...
505,Wyoming,2017-18,30409,29945,464
506,Wyoming,2018-19,30058,30020,38
507,Wyoming,2019-20,29931,29678,253
508,Wyoming,2020-21,28456,27871,585


In [213]:
states = state_enrollment['State'].unique()

total_change_pct_dict = {state: [] for state in states}
baseline_enrollment_dict = {state: [] for state in states}

for state in states:
    mask = state_trends['State'] == state
    df = state_trends[state_trends['State'] == state].copy()
    year_baseline = '2012-13'
    state_baseline = df[df['Year'] == year_baseline]['Total_Enrollment'].values[0]
    baseline_enrollment_dict[state].append(state_baseline)
    total_change_pct_dict[state].append(((state_trends[state_trends['State'] == state]['Total_Enrollment'].values[9] - state_trends[state_trends['State'] == state]['Total_Enrollment'].values[0])/state_baseline) * 100)
    df['enrollment_index'] = (df['Total_Enrollment'] / state_baseline) * 100
    df['yoy_change'] = df['Total_Enrollment'].pct_change() * 100
    state_trends.loc[mask, 'enrollment_index'] = df['enrollment_index']
    state_trends.loc[mask, 'yoy_change'] = df['yoy_change']

state_trends

,State,Year,Total_Enrollment,Total_Pub_Under,Total_Priv_Under,enrollment_index,yoy_change
0,Alabama,2012-13,265917,216535,49382,100.000000,NaN
1,Alabama,2013-14,261188,213669,47519,98.221626,-1.778374
2,Alabama,2014-15,259630,212458,47172,97.635728,-0.596505
3,Alabama,2015-16,257650,212968,44682,96.891135,-0.762624
4,Alabama,2016-17,258008,216115,41893,97.025764,0.138948
...,...,...,...,...,...,...,...
505,Wyoming,2017-18,30409,29945,464,86.627924,-1.227791
506,Wyoming,2018-19,30058,30020,38,85.628009,-1.154264
507,Wyoming,2019-20,29931,29678,253,85.266217,-0.422516
508,Wyoming,2020-21,28456,27871,585,81.064296,-4.928001


In [214]:
avg_change_pct_dict = {state: [] for state in states}
slope_dict = {state: [] for state in states}
intercept_dict = {state: [] for state in states}
ci_slope_dict = {state: [] for state in states}
r_squared_dict = {state: [] for state in states}
p_value_dict = {state: [] for state in states}


state_years = np.arange(len(state_trends[state_trends['State'] == 'Alabama']))

for state in states:
    slope,intercept, r_value, p_value, std_err = stats.linregress(state_years, state_trends[state_trends['State'] == state]['enrollment_index'])
    ci_slope = t.interval(0.95, len(state_years)-1, loc=slope, scale=std_err)
    slope_dict[state].append(slope)
    intercept_dict[state].append(intercept)
    ci_slope_dict[state].append(ci_slope)
    r_squared_dict[state].append(r_value**2)
    p_value_dict[state].append(p_value)
    avg_change_pct_dict[state].append(state_trends[state_trends['State'] == state]['yoy_change'].mean())

In [215]:
significant = []
insignificant = []
for state in states:
    print("STATE: ", state)
    print('Baseline: ', baseline_enrollment_dict.get(state)[0])
    print('2021-22: ', state_trends[state_trends['State'] == state]['Total_Enrollment'].values[9])
    print('Total Change %: ', total_change_pct_dict[state])
    print('Avg Change %: ', avg_change_pct_dict.get(state)[0])
    print('Slope: ', slope_dict.get(state)[0])
    print('Intercept: ', intercept_dict.get(state)[0])
    print('95% CI: ', ci_slope_dict.get(state)[0])
    print('R Squared: ', r_squared_dict.get(state)[0])
    print('P-Value: ', p_value_dict.get(state)[0])
    print('_______________________________________')
    print('')
    if p_value_dict.get(state)[0] < 0.05:
        significant.append(state)
    else:
        insignificant.append(state)

STATE:  Alabama
Baseline:  265917
2021-22:  239392
Total Change %:  [np.float64(-9.974916985375136)]
Avg Change %:  -1.1460086297790484
Slope:  -0.9769282076054494
Intercept:  100.26594080791443
95% CI:  (np.float64(-1.3821679751514382), np.float64(-0.5716884400594606))
R Squared:  0.7880255359690876
P-Value:  0.0006062015452915853
_______________________________________

STATE:  Alaska
Baseline:  30018
2021-22:  18943
Total Change %:  [np.float64(-36.894529948697446)]
Avg Change %:  -4.862862039150015
Slope:  -5.122542636034542
Intercept:  108.68872615824444
95% CI:  (np.float64(-6.226079005115164), np.float64(-4.0190062669539195))
R Squared:  0.9323561356925458
P-Value:  5.886735958030527e-06
_______________________________________

STATE:  Arizona
Baseline:  621610
2021-22:  501925
Total Change %:  [np.float64(-19.25403387976384)]
Avg Change %:  -2.291436690024194
Slope:  -2.1266457142327866
Intercept:  94.69002926418348
95% CI:  (np.float64(-3.0821704103354923), np.float64(-1.17112

In [217]:
print('Significant: ', significant)
print('Not Significant: ', insignificant)

Significant:  ['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'Connecticut', 'District of Columbia', 'Florida', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']
Not Significant:  ['California', 'Colorado', 'Delaware', 'Georgia', 'Nevada', 'Texas']


The states that show not statistically significant declining trend are ['California', 'Colorado', 'Delaware', 'Georgia', 'Nevada', 'Texas']. This as to do with their p value being greater than 0.05.

In [219]:
for state in significant:
    if slope_dict.get(state)[0] > 0:
        print(state + ': ', slope_dict.get(state)[0])

District of Columbia:  1.5156406938205302
Idaho:  1.6604377727889996
New Hampshire:  16.603854933943296
Utah:  5.219142158960169


Of the states with significant trends, only 4 have a positive trend, being D.C., Idaho, New Hampshire and Utah.

#### Visualization

In [221]:
## Positive Enrollment Trend
fig = go.Figure()

for state in ['District of Columbia', 'Idaho', 'New Hampshire', 'Utah']:
    fig.add_trace(go.Scatter(x=state_trends[state_trends['State'] == state]['Year'], y=state_trends[state_trends['State'] == state]['enrollment_index'], mode='lines', name=state))

fig.update_layout(title='Enrollment Index Over Time', xaxis_title='Year', yaxis_title='Total Enrollment')
fig.show()

In [223]:
## Negative Enrollment Trend
fig = go.Figure()

for state in states:
    if state not in ['District of Columbia', 'Idaho', 'New Hampshire', 'Utah']:
        fig.add_trace(go.Scatter(x=state_trends[state_trends['State'] == state]['Year'], y=state_trends[state_trends['State'] == state]['enrollment_index'], mode='lines', name=state))

fig.update_layout(title='Enrollment Index Over Time', xaxis_title='Year', yaxis_title='Total Enrollment')
fig.show()

In [224]:
## Not Significant Trends
fig = go.Figure()

for state in insignificant:
    fig.add_trace(go.Scatter(x=state_trends[state_trends['State'] == state]['Year'], y=state_trends[state_trends['State'] == state]['enrollment_index'], mode='lines', name=state))

fig.update_layout(title='Enrollment Index Over Time', xaxis_title='Year', yaxis_title='Total Enrollment')
fig.show()

**Question** which states have statistically significant declining trends?

**Answer** 'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'Connecticut', 'Florida', 'Hawaii', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming'

These states have statistically significant negative trends making them definitively decreasing for a cause other than randomness.

---------

**sub-question** Which states show statistically significant growth trends?

Using the Mann-Kendall test for this

----

**sub-question** What is the effect size of enrollment changes (Cohen's d)?

------

**sub-question** Are there inflection points where enrollment trends shifted direction?

-----